In [0]:
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
# --- Automated Pipeline Trigger Logic ---
# This script automatically identifies the target month based on a 6-month 
# offset from the current system date. It validates the presence of the 
# corresponding Parquet file in the 'Landing' directory. If the file is found, 
# the pipeline proceeds; otherwise, it stops to prevent processing errors.
# This ensures a robust, automated workflow without manual code updates.

# Calculate the target month (Current Date - 6 months)
target_date = date.today() - relativedelta(months=6)
target_month = target_date.strftime("%Y-%m")

# Define the path to the expected Parquet file
base_path = f"/Volumes/nyctaxi/00_landing/data_sources/nyctaxi_yellow/{target_month}"
local_path = f"{base_path}/yellow_tripdata_{target_month}.parquet"

try:
    # Verify if the file exists in the Landing zone
    dbutils.fs.ls(local_path)
    
    # Enable downstream tasks if the file is available
    dbutils.jobs.taskValues.set(key="continue_downstream", value="yes")
    print(f"Data for {target_month} successfully validated. Pipeline proceeding.")
    
except Exception as e:
    # Disable downstream tasks if the file is missing
    dbutils.jobs.taskValues.set(key="continue_downstream", value="no")
    print(f"Validation failed: Data for {target_month} not found. Pipeline halted.")